In [1]:
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")
file_path="the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x13b0bbc0d00>)

In [2]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("总字符数：", len(raw_text))

# 打印前100个字符
print(raw_text[:99])

总字符数： 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [3]:
import re

# 有的单词和标点无法分离
text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [4]:
# 修改正则表达式使其分离，但是列表中依然含有空白字符
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [5]:
# 过滤空白字符
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [6]:
# 对特殊字符进行分词
text = "Hello, world. Is this-- a test?" 
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()] 
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [7]:
# 对导入的小说进行分词
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))
print(preprocessed[:30])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [8]:
# 构建词汇表(利用集合去重)
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

vocab = {token:integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

1130
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [9]:
# 实现简单文本分词器（无法识别词汇表之外的词汇）
class SimpleTokenizerV1:
    def __init__(self, vocab):
        # 将词汇表作为类属性存储，以便在 encode 方法和 decode 方法中访问
        self.str_to_int = vocab
        # 创建逆向词汇表，将词元 ID 映射回原始文本词元
        self.int_to_str = {i:s for s,i in vocab.items()}
        
    # 文本 → Token ID
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    # Token ID → 文本
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)
        return text

In [10]:
tokenizer = SimpleTokenizerV1(vocab) 
text = """"It's the last he painted, you know," 
 Mrs. Gisburn said with pardonable pride.""" 
ids = tokenizer.encode(text) 
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [11]:
text = "Hello, do you like tea?" 
print(tokenizer.encode(text))

KeyError: 'Hello'

In [12]:
# 在词汇表中加入2个特殊词元
all_tokens = sorted(list(set(preprocessed))) 
all_tokens.extend(["<|endoftext|>", "<|unk|>"]) 
vocab = {token:integer for integer,token in enumerate(all_tokens)} 
print(len(vocab.items()))

1132


In [13]:
# 能够处理未知单词的文本分词器
class SimpleTokenizerV2:
    def __init__(self, vocab):
        # 将词汇表作为类属性存储，以便在 encode 方法和 decode 方法中访问
        self.str_to_int = vocab
        # 创建逆向词汇表，将词元 ID 映射回原始文本词元
        self.int_to_str = {i:s for s,i in vocab.items()}
        
    # 文本 → Token ID
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # 用<|unk|>替换未知单词
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    # Token ID → 文本
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)
        return text

In [14]:
text1 = "Hello, do you like tea?" 
text2 = "In the sunlit terraces of the palace." 
text = " <|endoftext|> ".join((text1, text2)) 
print(text)

tokenizer = SimpleTokenizerV2(vocab) 
print("\n")
print(tokenizer.encode(text))
print("\n")
print(tokenizer.decode(tokenizer.encode(text)))

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [15]:
from importlib.metadata import version
import tiktoken
print(version("tiktoken"))

0.12.0


In [16]:
tokenizer = tiktoken.get_encoding("gpt2")
text = ( 
 "Hello, do you like tea? <|endoftext|> In the sunlit terraces" 
 "of someunknownPlace." 
) 
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"}) 
print(integers)
print("\n")
strings = tokenizer.decode(integers) 
print(strings)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [17]:
# Exercise 2.1
word = ("Akwirw ier")
ids = tokenizer.encode(word, allowed_special={"<|endoftext|>"})
print(ids)

string = tokenizer.decode(ids)
print(string)

[33901, 86, 343, 86, 220, 959]
Akwirw ier


In [18]:
with open('the-verdict.txt', 'r', encoding="utf-8") as f:
    raw_text = f.read() 
enc_text = tokenizer.encode(raw_text) 
print(len(enc_text))

5145


In [19]:
enc_sample = enc_text[50:]

# 使用滑动窗口实现数据加载器
context_size = 4 # 上下文大小决定了输入中包含多少词元
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [20]:
# for 循环实现窗口移动
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "-->", desired)

[290] --> 4920
[290, 4920] --> 2241
[290, 4920, 2241] --> 287
[290, 4920, 2241, 287] --> 257


In [21]:
# 窗口中显示词元
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "-->", tokenizer.decode([desired]))

 and -->  established
 and established -->  himself
 and established himself -->  in
 and established himself in -->  a


In [22]:
# 一个用于批处理输入和目标的数据集
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride): # stride 决定批次之间输入的位移量
        self.input_ids = []
        self.target_ids = []

        # 对所有文本进行分词
        token_ids = tokenizer.encode(txt)

        # 使用滑动窗口将文本划分为长度为 max_length 的重叠序列
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i: i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    # 返回数据集的总行数
    def __len__(self):
        return len(self.input_ids)

    # 返回数据的指定行
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# 用于批量生成输入-目标对的数据加载器
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    
    # 初始化分词器
    tokenizer = tiktoken.get_encoding("gpt2")

    # 创建数据集
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    
    # 如果 drop_last 为 True 且批次大小小于指定的 batch_size，则会删除最后一批以防止训练期间出现损失剧增
    # num_workers 用于设定预处理的 CPU 进程数
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader

In [23]:
# 测试
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)

# 将 dataloader 转换为 Python 迭代器，以通过 Python 内置的 next() 函数获取下一个条目
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
print("\n")

second_batch = next(data_iter)
print(second_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [24]:
#Exercise 2.2
## max_length=2, stride=2
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=2, stride=2, shuffle=False)

# 将 dataloader 转换为 Python 迭代器，以通过 Python 内置的 next() 函数获取下一个条目
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
print("\n")

second_batch = next(data_iter)
print(second_batch)

[tensor([[ 40, 367]]), tensor([[ 367, 2885]])]


[tensor([[2885, 1464]]), tensor([[1464, 1807]])]


In [25]:
## max_length=8, stride=2
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=8, stride=1, shuffle=False)

# 将 dataloader 转换为 Python 迭代器，以通过 Python 内置的 next() 函数获取下一个条目
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
print("\n")

second_batch = next(data_iter)
print(second_batch)

[tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]


[tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]]), tensor([[ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138]])]


In [26]:
# 以大于 1 的 batch_size 采样
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

# 将 dataloader 转换为 Python 迭代器，以通过 Python 内置的 next() 函数获取下一个条目
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("input:\n", inputs)
print("\nTarget:\n", targets)

input:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Target:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [27]:
# token ID → embedding vector
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer)
print("\n")
print(embedding_layer.weight)

Embedding(6, 3)


Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [28]:
# 应用到一个 tokenID
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [29]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [30]:
vocab_size = 50257 
output_dim = 256 
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

# 实例化上面建立的数据加载器
max_length = 4 
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False) 
data_iter = iter(dataloader) 
inputs, targets = next(data_iter) 
print("Token IDs:\n", inputs) 
print("\nInputs shape:\n", inputs.shape)

print("\n")

# 使用嵌入层将这些词元 ID 嵌入 256 维的向量
token_embeddings = token_embedding_layer(inputs) 
print(token_embeddings.shape)

# 获取 GPT 模型所采用的绝对位置嵌入
context_length = max_length 
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim) 
pos_embeddings = pos_embedding_layer(torch.arange(context_length)) 
print("\n")
print(pos_embeddings.shape)

print("\n")

# 将这些向量直接添加到词元嵌入中
input_embeddings = token_embeddings + pos_embeddings 
print(input_embeddings.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


torch.Size([8, 4, 256])


torch.Size([4, 256])


torch.Size([8, 4, 256])
